In [126]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ComponentEmbedding(nn.Module):
    """Embeds circuit components into a latent space"""
    
    def __init__(self, num_component_types, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_component_types, embedding_dim)
        
    def forward(self, component_types):
        """
        Args:
            component_types: Tensor of component type indices [batch_size, num_nodes] or [num_nodes]
        Returns:
            Component embeddings [batch_size, num_nodes, embedding_dim] or [num_nodes, embedding_dim]
        """
        return self.embedding(component_types)


class GraphAttention(nn.Module):
    """Multi-head graph attention layer for processing the adjacency matrix with edge bias."""
    
    def __init__(self, input_dim, output_dim, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = output_dim // num_heads
        assert output_dim % num_heads == 0, "output_dim must be divisible by num_heads"
        
        # Query, Key, Value projections
        self.query = nn.Linear(input_dim, output_dim)
        self.key = nn.Linear(input_dim, output_dim)
        self.value = nn.Linear(input_dim, output_dim)
        
        # Output projection
        self.output_proj = nn.Linear(output_dim, output_dim)
        
        # Edge feature integration: projects a single scalar edge feature to each head's bias
        self.edge_proj = nn.Linear(1, self.num_heads)
        
    def forward(self, node_features, adjacency):
        """
        Args:
            node_features: Node feature tensor of shape [batch_size, num_nodes, input_dim]
            adjacency: Adjacency matrix tensor of shape [batch_size, num_nodes, num_nodes]
        Returns:
            Updated node features of shape [batch_size, num_nodes, output_dim]
        """
        batch_size = node_features.size(0)
        num_nodes = node_features.size(1)

        # Project inputs to queries, keys, and values and then reshape for multi-head attention
        # Each projection now yields shape: [batch_size, num_nodes, output_dim]
        q = self.query(node_features).view(batch_size, num_nodes, self.num_heads, self.head_dim)
        k = self.key(node_features).view(batch_size, num_nodes, self.num_heads, self.head_dim)
        v = self.value(node_features).view(batch_size, num_nodes, self.num_heads, self.head_dim)
        
        # Rearrange for attention: [batch_size, num_heads, num_nodes, head_dim]
        q = q.permute(0, 2, 1, 3)
        k = k.permute(0, 2, 3, 1)  # for dot-product attention, key is transposed on last two dims
        v = v.permute(0, 2, 1, 3)
        
        # Compute raw attention scores with scaling
        attn_scores = torch.matmul(q, k) / (self.head_dim ** 0.5)  # [batch_size, num_heads, num_nodes, num_nodes]
        
        # Incorporate edge information:
        # Expecting adjacency shape [batch_size, num_nodes, num_nodes]
        # Unsqueeze last dimension and project edge feature per head
        edge_bias = self.edge_proj(adjacency.unsqueeze(-1))  # [batch_size, num_nodes, num_nodes, num_heads]
        edge_bias = edge_bias.permute(0, 3, 1, 2)  # [batch_size, num_heads, num_nodes, num_nodes]
        attn_scores = attn_scores + edge_bias
        # Mask out non-adjacent nodes:
        mask = (adjacency == 0).unsqueeze(1)  # [batch_size, 1, num_nodes, num_nodes]
        # Mask last row and column for self-attention
        # mask[:, :, -1, :] = True
        # mask[:, :, :, -1] = True

        attn_scores = attn_scores.masked_fill(mask, float('-inf'))
        
        # Compute normalized attention weights
        attn_weights = F.softmax(attn_scores, dim=-1)
        
        # Weighted sum over values
        out = torch.matmul(attn_weights, v)  # [batch_size, num_heads, num_nodes, head_dim]
        
        # Reshape back: first bring num_nodes back to dim 1
        out = out.permute(0, 2, 1, 3).contiguous()
        out = out.view(batch_size, num_nodes, self.num_heads * self.head_dim)  # [batch_size, num_nodes, output_dim]
        
        # Final projection
        out = self.output_proj(out)
        
        return out


class CircuitGraphTransformer(nn.Module):
    """Circuit graph transformer model using graph attention layers"""
    
    def __init__(self, num_component_types, hidden_dim=256, num_layers=6, num_heads=8, dropout=0.1):
        super().__init__()
        
        # Component embedding layer
        self.component_embedding = ComponentEmbedding(num_component_types, hidden_dim)
        
        # Positional encoding: supports up to 330 nodes
        self.position_embedding = nn.Parameter(torch.zeros(1, 330, hidden_dim))
        
        # Build a sequence of transformer layers with attention and feed-forward components
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'attention': GraphAttention(hidden_dim, hidden_dim, num_heads=num_heads),
                'norm1': nn.LayerNorm(hidden_dim),
                'ffn': nn.Sequential(
                    nn.Linear(hidden_dim, hidden_dim * 4),
                    nn.ReLU(),
                    nn.Linear(hidden_dim * 4, hidden_dim),
                    nn.Dropout(dropout)
                ),
                'norm2': nn.LayerNorm(hidden_dim)
            })
            for _ in range(num_layers)
        ])
        
        # Edge predictor network that uses concatenated node pairs to predict edge probabilities
        self.edge_predictor = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
        
    def forward(self, component_types, adjacency_matrix):
        """
        Args:
            component_types: Component type indices, shape [num_nodes] or [batch_size, num_nodes]
            adjacency_matrix: Adjacency matrix, shape [num_nodes, num_nodes] or [batch_size, num_nodes, num_nodes]
        Returns:
            Node features tensor: [batch_size, num_nodes, hidden_dim]
        """
        # Add batch dimension if inputs are for a single graph.
        if component_types.dim() == 1:
            component_types = component_types.unsqueeze(0)  # [1, num_nodes]
        if adjacency_matrix.dim() == 2:
            adjacency_matrix = adjacency_matrix.unsqueeze(0)  # [1, num_nodes, num_nodes]
        
        # Obtain component embeddings and add positional embeddings.
        node_features = self.component_embedding(component_types)  # [batch_size, num_nodes, hidden_dim]
        num_nodes = node_features.size(1)
        node_features = node_features + self.position_embedding[:, :num_nodes, :]
        
        # Process through each transformer layer
        for layer in self.layers:
            attn_output = layer['attention'](node_features, adjacency_matrix)
            node_features = layer['norm1'](node_features + attn_output)
            
            ffn_output = layer['ffn'](node_features)
            node_features = layer['norm2'](node_features + ffn_output)
        
        return node_features
        
    def predict_new_edges(self, component_types, adjacency_matrix, new_component_type):
        """
        Predicts connection probabilities between a new component and existing components.
        
        Args:
            component_types: Component type indices, shape [num_nodes] or [batch_size, num_nodes]
            adjacency_matrix: Adjacency matrix, shape [num_nodes, num_nodes] or [batch_size, num_nodes, num_nodes]
            new_component_type: Type index of the new component to add
        Returns:
            Edge probabilities for the new component: Tensor of shape [num_nodes]
        """
        # Get node embeddings
        node_features = self.forward(component_types, adjacency_matrix)
        # If using a single graph, remove the batch dimension.
        if node_features.size(0) == 1:
            node_features = node_features[0]  # [num_nodes, hidden_dim]
        
        # Get embedding for the new component, shape [1, hidden_dim]
        new_component_embedding = self.component_embedding(
            torch.tensor([new_component_type], device=component_types.device)
        )
        
        edge_probs = []
        for i in range(node_features.size(0)):
            # Concatenate features of existing node and new component
            pair_features = torch.cat([node_features[i], new_component_embedding[0]], dim=-1)
            edge_prob = self.edge_predictor(pair_features)  # Output shape: [1]
            edge_probs.append(edge_prob)
        
        return torch.cat(edge_probs)
  

In [127]:

class CircuitGraphPredictor:
    """High-level interface for the circuit graph prediction model"""
    
    def __init__(self, model_path=None, num_component_types=830, hidden_dim=256):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        #self.device = torch.device('cpu')
        self.model = CircuitGraphTransformer(
            num_component_types=num_component_types,
            hidden_dim=hidden_dim
        ).to(self.device)
        
        if model_path:
            self.model.load_state_dict(torch.load(model_path, map_location=self.device))
        
    def predict_new_connections(self, components, adjacency_matrix, new_component_type, threshold=0.5):
        """
        Predicts connections for a new component to be added to the circuit.
        
        Args:
            components: List of component type indices
            adjacency_matrix: Current adjacency matrix as a numpy array or list [n x n]
            new_component_type: Type index of the new component
            threshold: Threshold probability to decide on edge creation
        
        Returns:
            New connections for the new component: Numpy array of shape [num_nodes] with 0/1 values.
        """
        # Convert inputs to tensors
        component_types = torch.tensor(components, dtype=torch.long).to(self.device)
        adj_matrix = torch.tensor(adjacency_matrix, dtype=torch.float).to(self.device)
        
        # Predict edge probabilities using the model without gradient calculations.
        with torch.no_grad():
            edge_probs = self.model.predict_new_edges(
                component_types, adj_matrix, new_component_type
            )
        
        # Apply the threshold to obtain binary edge predictions
        new_connections = (edge_probs >= threshold).float().cpu().numpy()
        
        return new_connections
    


In [128]:

# Example usage:
def create_dataset_from_circuits(circuit_graphs, component_libraries):
    """
    Create a dataset from a collection of circuit graphs for training the model.
    
    Args:
        circuit_graphs: List of (component_list, adjacency_matrix) tuples.
        component_libraries: Dictionary mapping component names to type indices.
        
    Returns:
        Dataset for training the model.
    """
    # Implementation depends on specific data format.
    pass


def main():
    # Initialize predictor with 50 component types.
    predictor = CircuitGraphPredictor(num_component_types=50)
    
    # Example components and corresponding adjacency matrix for a single graph.
    components = [0, 1, 2, 3]  # Component type indices
    adjacency_matrix = [
        [0, 1, 0, 1],  # Component 0 connects to 1 and 3
        [1, 0, 1, 0],  # Component 1 connects to 0 and 2
        [0, 1, 0, 1],  # Component 2 connects to 1 and 3
        [1, 0, 1, 0]   # Component 3 connects to 0 and 2
    ]
    
    # Predict new connections for a new component of type 4.
    new_component_type = 4
    new_connections = predictor.predict_new_connections(
        components, adjacency_matrix, new_component_type
    )
    
    print(f"Predicted connections for new component: {new_connections}")
    
    # The new adjacency matrix would include the new row and column corresponding to new_connections.
    # For example:
    # [
    #   [0, 1, 0, 1, new_connections[0]],
    #   [1, 0, 1, 0, new_connections[1]],
    #   [0, 1, 0, 1, new_connections[2]],
    #   [1, 0, 1, 0, new_connections[3]],
    #   [new_connections[0], new_connections[1], new_connections[2], new_connections[3], 0]
    # ]

    


In [129]:
from Circuits import Circuits
def dataset():
    graphdataset,textdataset = Circuits().data_lodder_withoutpading()
    dataset=list(zip(graphdataset,textdataset))
    return dataset
#dataloader=dataset()

In [ ]:
# New training loop
num_epochs = 5
learning_rate = 5e-4

predictor = CircuitGraphPredictor(num_component_types=895)

# Reinitialize the optimizer with a new learning rate
optimizer = torch.optim.Adam(predictor.model.parameters(), lr=learning_rate)
loss_fn = nn.BCELoss()

for epoch in range(num_epochs):
    total_loss = 0
    for adjacency_matrix, component_types  in dataloader:
        n=adjacency_matrix.shape[0]
        component_types = component_types.to(predictor.device)
        adjacency_matrix = adjacency_matrix.to(predictor.device)

        for t in range(2, n):
            input_adj = adjacency_matrix[:t, :t]
            input_feats = component_types[:t] 
        
            # Get node embeddings
            node_features = predictor.model(component_types, adjacency_matrix)
            
            # Calculate pairwise edge predictions (ignoring self-loops)
            edge_preds = []
            num_nodes = node_features.size(1)
            for i in range(1,t):
                        pair_features = torch.cat([node_features[:, i, :], node_features[:, t-1, :]], dim=-1)
                        edge_prob = predictor.model.edge_predictor(pair_features).squeeze(-1)
                        edge_preds.append(edge_prob)
                
            edge_preds = torch.cat(edge_preds)  # Should align with flattened target_edges
            
            loss = loss_fn(edge_preds, adjacency_matrix[t,:t-1].view(-1))
            
                            # Backward pass and update
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader):.4f}")

KeyboardInterrupt: 